In [1]:
from google.adk.agents.llm_agent import Agent
from dotenv import load_dotenv
import google.auth
from google.adk.telemetry import google_cloud
from google.adk.telemetry.setup import maybe_set_otel_providers
import os
import asyncio
from datetime import datetime, timezone
from typing import Optional

import google.auth
from google.adk.agents import LlmAgent            # canonical import in ADK 2.x
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import InMemoryRunner
from google.adk.telemetry import google_cloud
from google.adk.telemetry.setup import maybe_set_otel_providers
from google.cloud import logging as gcp_logging
from google.genai import types
load_dotenv()

True

In [2]:
# Cell 3

# Resolve credentials and project ID explicitly
credentials, project_id = google.auth.default()
print(f"Project ID: {project_id}")

# Build OTel resource with the project ID so Cloud Trace knows where to export
resource = google_cloud.get_gcp_resource(project_id=project_id)

# Get GCP exporters configuration
hooks = google_cloud.get_gcp_exporters(
    enable_cloud_tracing=True,
    google_auth=(credentials, project_id),
)

# Initialize and set global OTel providers
maybe_set_otel_providers(otel_hooks_to_setup=[hooks], otel_resource=resource)

print("✅ Cloud Trace initialized")

Project ID: dev-mq-tech-transfer
✅ Cloud Trace initialized


In [3]:
APP_NAME = "city_time_app"
USER_ID = "prakash.kc"

## Initialize Logger

In [4]:
gcp_logger = gcp_logging.Client().logger("llm-calls")

## Initialize Callbacks

In [5]:
# ── Callbacks ────────────────────────────────────────────────────────────────
def before_model_callback(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> LlmResponse | None:
    """Log user email + query before the LLM is called."""

    # ADK versions may expose session as either callback_context.session_id or callback_context.session.id
    session_id = getattr(callback_context, "session_id", None)
    if not session_id:
        session = getattr(callback_context, "session", None)
        session_id = getattr(session, "id", None)

    # ADK 2.x: messages live in llm_request.contents (list[types.Content])
    user_text = ""
    for content in reversed(llm_request.contents or []):
        if content.role == "user":
            user_text = " ".join(
                part.text
                for part in (content.parts or [])
                if getattr(part, "text", None)
            )
            break

    gcp_logger.log_struct(
        {
            "event":      "llm_request",
            "timestamp":  datetime.now(timezone.utc).isoformat(),
            "user_email": callback_context.state.get("user:email", USER_ID),
            "session_id": session_id,
            "user_query": user_text,
        },
        severity="INFO",
    )
    return None  # None -> proceed with the real LLM call


def after_model_callback(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> LlmResponse | None:
    """Log the LLM response text after the model replies."""

    session_id = getattr(callback_context, "session_id", None)
    if not session_id:
        session = getattr(callback_context, "session", None)
        session_id = getattr(session, "id", None)

    # ADK response shape varies by version: handle both candidates[0].content.parts and content.parts
    parts = []
    candidates = getattr(llm_response, "candidates", None)
    if candidates:
        first_content = getattr(candidates[0], "content", None)
        parts = getattr(first_content, "parts", None) or []
    else:
        content = getattr(llm_response, "content", None)
        parts = getattr(content, "parts", None) or []

    response_text = " ".join(
        part.text
        for part in parts
        if getattr(part, "text", None)
    )

    # Skip logging intermediate tool-call turns (no text in them)
    if not response_text:
        return None

    gcp_logger.log_struct(
        {
            "event":        "llm_response",
            "timestamp":    datetime.now(timezone.utc).isoformat(),
            "user_email":   callback_context.state.get("user:email", USER_ID),
            "session_id":   session_id,
            "llm_response": response_text,
        },
        severity="INFO",
    )
    return None  # None -> pass the real response through unchanged

In [6]:
# from google.adk.agents import Agent

# root_agent = Agent(
#     name="capital_agent",
#     model="gemini-2.5-flash",
#     description="Answers geography questions.",
#     instruction="Answer the user's question directly and concisely.",
# )

# print("✅ Agent created")

# Mock tool implementation
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city."""
    return {"status": "success", "city": city, "time": "10:30 AM"}

root_agent = Agent(
    model=os.environ.get("MODEL_NAME", "gemini-2.5-flash"),
    name='root_agent',
    description="Tells the current time in a specified city.",
    instruction="You are a helpful assistant that tells the current time in cities. Use the 'get_current_time' tool for this purpose.",
    tools=[get_current_time],
    before_model_callback=before_model_callback,   # ← added
    after_model_callback=after_model_callback,     # ← added
)

In [7]:
# Cell 5
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part

async def ask(question: str):
    runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)

    session = await runner.session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
    )

    user_message = Content(parts=[Part(text=question)])

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print("Agent:", part.text)



In [8]:
await ask("What is the time in London?")

c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


Agent: The current time in London is 10:30 AM.
